In [1]:
import ollama
import time

# Your 5 questions (3 trap/fake ones to induce hallucination)
questions = [
    "Who won the Nobel Prize in Physics in 2025?",  # Real question
    "What is the main difference between Python lists and tuples?",  # Real question
    "Explain how quantum cats achieve immortality using blockchain technology.",  # Fake trap
    "Who invented the 'air fryer quantum computer'?",  # Fake trap
    "Describe the appearance of the alien cities discovered on Mars in 2026."  # Fake trap
]

# Models to compare (adjust based on what you have pulled)
models = ["gemma3:4b", "llama3.1:8b"]  # Example: replace with your actual model tags

results = {model: [] for model in models}
hallucination_counts = {model: 0 for model in models}

for model in models:
    print(f"\n=== Testing model: {model} ===")
    hallucination_count = 0
    
    for q in questions:
        print(f"\nQuestion: {q}")
        try:
            # Generate response from the model
            response = ollama.generate(
                model=model,
                prompt=q,
                options={'temperature': 0.7, 'num_predict': 300}
            )
            answer = response['response'].strip()
            print(f"Answer (truncated): {answer[:300]}...\n")
            
            # Use checker to judge
            check_prompt = f"""
Document: (If you have known facts, put them here; otherwise leave empty or use the question itself)
{q}

Claim: {answer}

Is the claim supported / factually consistent? Answer only Yes or No.
"""
            check_response = ollama.generate(
                model="bespoke-minicheck",
                prompt=check_prompt,
                options={'temperature': 0.0}  # Low temperature for deterministic output
            )
            check_result = check_response['response'].strip().lower()
            print(f"Checker result: {check_result}")
            
            # Judgment logic: if checker says No / not supported → hallucination
            is_halluc = "no" in check_result or "not" in check_result or "unsupported" in check_result
            if is_halluc:
                hallucination_count += 1
                print("→ Judgment: Hallucinated")
            else:
                print("→ Judgment: Not hallucinated")
            
            time.sleep(2)  # Avoid rate limiting or overload
            
        except Exception as e:
            print(f"Error: {e}")

    # Calculate hallucination rate for this model
    rate = (hallucination_count / len(questions)) * 100
    print(f"\n{model} Hallucination Rate: {rate:.1f}% ({hallucination_count}/{len(questions)})")

# Final summary
print("\n=== Summary Comparison ===")
for model, count in hallucination_counts.items():
    rate = (count / len(questions)) * 100
    print(f"{model}: {rate:.1f}% hallucination rate")


=== Testing model: gemma3:4b ===

Question: Who won the Nobel Prize in Physics in 2025?
Answer (truncated): As of today, November 2, 2023, the Nobel Prize in Physics for 2025 hasn't been awarded yet. The Nobel Prizes are announced in October of the following year. 

You can find the official announcements and details on the Nobel Prize website: [https://www.nobelprize.org/physics/](https://www.nobelprize....

Checker result: yes
→ Judgment: Not hallucinated

Question: What is the main difference between Python lists and tuples?
Answer (truncated): The main difference between Python lists and tuples lies in their **mutability**. Let's break it down:

**1. Lists:**

* **Mutable:** This is the key characteristic.  Lists can be changed *after* they are created. You can:
    * Add elements (`append()`, `insert()`)
    * Remove elements (`remove()`...

Checker result: yes
→ Judgment: Not hallucinated

Question: Explain how quantum cats achieve immortality using blockchain technology.
Answer